In [1]:
pip install torch torchvision

In [2]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity

# tiny transformer model
model = nn.Transformer(
    d_model=128,
    nhead=4,
    num_encoder_layers=2,
    num_decoder_layers=2
)

src = torch.rand(10, 32, 128)
tgt = torch.rand(10, 32, 128)

# profile it
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    with_stack=True
) as prof:
    with record_function("model_forward"):
        output = model(src, tgt)

# export real trace
prof.export_chrome_trace("trace_0.json")
print(" First real trace generated!")
print(f"Operations recorded: {len(prof.key_averages())}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


✅ First real trace generated!
Operations recorded: 47


In [3]:
import json

# load the trace
with open("trace_0.json", "r") as f:
    trace = json.load(f)

# see what's inside
print("Keys in trace:", trace.keys())
print("Total events:", len(trace["traceEvents"]))
print("\nFirst 5 events:")
for event in trace["traceEvents"][:5]:
    print(event)

Keys in trace: dict_keys(['schemaVersion', 'deviceProperties', 'record_shapes', 'with_stack', 'trace_id', 'displayTimeUnit', 'baseTimeNanoseconds', 'traceEvents', 'traceName'])
Total events: 2613

First 5 events:
{'ph': 'X', 'cat': 'user_annotation', 'name': 'model_forward', 'pid': 3991, 'tid': 3991, 'ts': 6906560431836.557, 'dur': 238944.927, 'args': {'External id': 1, 'Record function id': 0, 'Ev Idx': 0}}
{'ph': 'X', 'cat': 'cpu_op', 'name': 'aten::linear', 'pid': 3991, 'tid': 3991, 'ts': 6906560436866.116, 'dur': 45349.966, 'args': {'External id': 2, 'Sequence number': 0, 'Fwd thread id': 0, 'Record function id': 0, 'Concrete Inputs': ['', '', ''], 'Input type': ['float', 'float', 'float'], 'Input Strides': [[4096, 128, 1], [128, 1], [1]], 'Input Dims': [[10, 32, 128], [384, 128], [384]], 'Ev Idx': 1}}
{'ph': 'X', 'cat': 'cpu_op', 'name': 'aten::view', 'pid': 3991, 'tid': 3991, 'ts': 6906560441319.673, 'dur': 1798.682, 'args': {'External id': 3, 'Sequence number': 0, 'Fwd thread id

In [5]:
# find critical path (longest path by duration)
def find_critical_path(G):
    for u, v in G.edges():
        G[u][v]['weight'] = G.nodes[u]['duration_ms']

    critical_path = nx.dag_longest_path(G, weight='weight')
    critical_path_length = nx.dag_longest_path_length(G, weight='weight')

    return critical_path, critical_path_length

critical_path, total_time = find_critical_path(G)

print(f"✅ Critical path found!")
print(f"Total step time: {total_time:.2f} ms")
print(f"Critical path length: {len(critical_path)} operations")
print(f"\nTop 5 bottleneck operations:")
bottlenecks = sorted(
    [(n, G.nodes[n]['duration_ms'], G.nodes[n]['name'])
     for n in critical_path],
    key=lambda x: x[1],
    reverse=True
)[:5]

for node_id, duration, name in bottlenecks:
    print(f"  {name}: {duration:.2f}ms")

✅ Critical path found!
Total step time: 534.93 ms
Critical path length: 1134 operations

Top 5 bottleneck operations:
  aten::linear: 45.35ms
  aten::scaled_dot_product_attention: 42.34ms
  aten::_scaled_dot_product_attention_math: 40.02ms
  aten::addmm: 33.85ms
  aten::_safe_softmax: 16.97ms


In [4]:
import json
import networkx as nx

def parse_trace(filepath):
    with open(filepath) as f:
        trace = json.load(f)

    # filter only cpu operations
    ops = [
        e for e in trace["traceEvents"]
        if e.get("cat") == "cpu_op" and "dur" in e
    ]

    # sort by timestamp
    ops = sorted(ops, key=lambda x: x["ts"])

    # build directed graph
    G = nx.DiGraph()

    for i, op in enumerate(ops):
        # classify node type
        node_type = "network" if "allreduce" in op["name"].lower() or "comm" in op["name"].lower() else "compute"

        # add node with features
        G.add_node(i,
            name=op["name"],
            type=node_type,
            duration_ms=op["dur"] / 1000,  # convert to ms
            timestamp=op["ts"]
        )

        # add edge from previous op (sequential dependency)
        if i > 0:
            G.add_edge(i-1, i)

    return G

G = parse_trace("trace_0.json")
print(f"✅ Graph built!")
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")
print(f"\nSample node:")
print(G.nodes[0])

✅ Graph built!
Nodes: 1134
Edges: 1133

Sample node:
{'name': 'aten::linear', 'type': 'compute', 'duration_ms': 45.349966, 'timestamp': 6906560436866.116}


In [6]:
def add_features(G):
    total_time = sum(G.nodes[n]['duration_ms'] for n in G.nodes())

    for n in G.nodes():
        duration = G.nodes[n]['duration_ms']

        # feature 1: compute ratio
        G.nodes[n]['compute_ratio'] = duration / total_time

        # feature 2: is on critical path
        G.nodes[n]['is_critical'] = 1 if n in critical_path else 0

        # feature 3: normalized duration
        max_dur = max(G.nodes[i]['duration_ms'] for i in G.nodes())
        G.nodes[n]['norm_duration'] = duration / max_dur

    return G

G = add_features(G)
print("✅ Features added!")
print("\nSample node with features:")
print(G.nodes[0])

✅ Features added!

Sample node with features:
{'name': 'aten::linear', 'type': 'compute', 'duration_ms': 45.349966, 'timestamp': 6906560436866.116, 'compute_ratio': 0.08477705098501838, 'is_critical': 1, 'norm_duration': 1.0}


In [7]:
!pip install torch-geometric -q

import torch
from torch_geometric.data import HeteroData

def to_pyg(G):
    data = HeteroData()

    # separate compute and network nodes
    compute_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'compute']
    network_nodes = [n for n in G.nodes() if G.nodes[n]['type'] == 'network']

    # compute node features
    compute_features = torch.tensor([
        [G.nodes[n]['duration_ms'],
         G.nodes[n]['compute_ratio'],
         G.nodes[n]['norm_duration'],
         G.nodes[n]['is_critical']]
        for n in compute_nodes
    ], dtype=torch.float)

    # network node features
    network_features = torch.tensor([
        [G.nodes[n]['duration_ms'],
         G.nodes[n]['compute_ratio'],
         G.nodes[n]['norm_duration'],
         G.nodes[n]['is_critical']]
        for n in network_nodes
    ], dtype=torch.float) if network_nodes else torch.zeros((0, 4))

    data['compute'].x = compute_features
    data['network'].x = network_features

    print(f"✅ PyG HeteroData created!")
    print(f"Compute nodes: {data['compute'].x.shape}")
    print(f"Network nodes: {data['network'].x.shape}")

    return data

data = to_pyg(G)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.9 MB/s eta 0:00:00
✅ PyG HeteroData created!
Compute nodes: torch.Size([1134, 4])
Network nodes: torch.Size([0, 4])
